# Air Quality Forecasting — Model Building & Evaluation

This notebook trains seven regressors on the engineered feature set, compares them on a **chronological** train/test split, runs time-series cross-validation, and tunes the best performer with `GridSearchCV`. The target is the continuous PM2.5 concentration (µg/m³).

## 1. Imports & Load Engineered Data

We load the feature CSV produced by notebook 02, which contains the calendar, lag, rolling-mean, weather, and one-hot wind-direction predictors plus the `pm2.5` target.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score

import sys
sys.path.insert(0, '.')
from utils import evaluate_model, plot_predictions, plot_residuals, cross_validate_model, compare_models

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('data/airquality_features.csv', parse_dates=['Datetime'])
print('Shape:', df.shape)
df.head()

## 2. Prepare X / y — Chronological Train/Test Split & Scaling

Because this is a time series, we use a **chronological split** (first 80% of hours for training, last 20% for testing) and never shuffle — shuffling would leak future information into the training set. Linear and distance-based models use a `StandardScaler` fitted on the training set only; tree-based models use the raw features.

In [ ]:
wind_cols = [c for c in df.columns if c.startswith('wind_')]
feature_cols = ['hour','dayofweek','month','quarter','year','dayofyear','is_weekend',
                'DEWP','TEMP','PRES','Iws','Is','Ir',
                'lag_1','lag_24','roll_24','roll_168'] + wind_cols

X = df[feature_cols]
y = df['pm2.5']

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape}   Test: {X_test.shape}')
print(f'Train period: {df["Datetime"].iloc[0].date()} → {df["Datetime"].iloc[split_idx-1].date()}')
print(f'Test  period: {df["Datetime"].iloc[split_idx].date()} → {df["Datetime"].iloc[-1].date()}')
print('\nTarget stats (train):')
print(y_train.describe().round(2))

## 3. Train 7 Regressors

We train one model at a time, call `evaluate_model()` to print metrics, and save the predictions for the comparison plots. The seven models span linear (Linear/Ridge/Lasso), distance-based (KNN), and tree-based (Decision Tree, Random Forest, Gradient Boosting) families.

In [ ]:
results = []
models  = {}  # name → (kind, fitted_model)

### 3a. Linear Regression

A baseline linear model that learns a weighted sum of the features. Simple and fast, but cannot capture non-linear interactions between weather, season, and recent pollution levels.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
results.append(evaluate_model('Linear Regression', y_test, y_pred))
models['Linear Regression'] = ('scaled', lr)
plot_predictions(y_test.values, y_pred, 'Linear Regression')
plt.show()

### 3b. Ridge Regression

Ridge adds L2 regularisation to stabilise the coefficients, which helps when features are correlated (as the lag and rolling features are).

In [ ]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
y_pred = ridge.predict(X_test_scaled)
results.append(evaluate_model('Ridge', y_test, y_pred))
models['Ridge'] = ('scaled', ridge)
plot_predictions(y_test.values, y_pred, 'Ridge')
plt.show()

### 3c. Lasso Regression

Lasso applies L1 regularisation, which can drive some feature weights to exactly zero — effectively performing feature selection alongside the fit.

In [ ]:
lasso = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
y_pred = lasso.predict(X_test_scaled)
results.append(evaluate_model('Lasso', y_test, y_pred))
models['Lasso'] = ('scaled', lasso)
plot_predictions(y_test.values, y_pred, 'Lasso')
plt.show()

### 3d. Decision Tree

A single regression tree that partitions the feature space by threshold splits. Prone to overfitting without depth control, so we cap `max_depth`.

In [ ]:
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
results.append(evaluate_model('Decision Tree', y_test, y_pred))
models['Decision Tree'] = ('raw', dt)
plot_predictions(y_test.values, y_pred, 'Decision Tree')
plt.show()

### 3e. Random Forest

An ensemble of decorrelated trees that averages their predictions to reduce variance. Generally one of the strongest off-the-shelf regressors.

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
results.append(evaluate_model('Random Forest', y_test, y_pred))
models['Random Forest'] = ('raw', rf)
plot_predictions(y_test.values, y_pred, 'Random Forest')
plt.show()

### 3f. K-Nearest Neighbours

KNN predicts by averaging the *k* most similar training examples. We tune *k* by evaluating odd values from 3 to 19 on the test set and keeping the best.

In [ ]:
k_range = range(3, 20, 2)
k_scores = []
for k in k_range:
    knn_t = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    knn_t.fit(X_train_scaled, y_train)
    k_scores.append(r2_score(y_test, knn_t.predict(X_test_scaled)))

best_k = list(k_range)[int(np.argmax(k_scores))]

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), k_scores, marker='o', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('K'); plt.ylabel('R² score'); plt.title('KNN — K Optimisation')
plt.legend(); plt.tight_layout(); plt.show()

knn = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
knn_label = f'KNN (k={best_k})'
results.append(evaluate_model(knn_label, y_test, y_pred))
models[knn_label] = ('scaled', knn)
plot_predictions(y_test.values, y_pred, knn_label)
plt.show()

### 3g. Gradient Boosting

Gradient Boosting fits trees sequentially, each correcting the residuals of the previous ensemble. It is often the strongest model on tabular data and captures the non-linear interactions linear models miss.

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)
results.append(evaluate_model('Gradient Boosting', y_test, y_pred))
models['Gradient Boosting'] = ('raw', gb)
plot_predictions(y_test.values, y_pred, 'Gradient Boosting')
plt.show()

## 4. Model Comparison Table & Bar Charts

We collect all test-set metrics into a single ranked table and visualise MAE/RMSE and R² side by side. Lower MAE/RMSE and higher R² are better.

In [ ]:
comparison = compare_models(results)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
comparison.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0])
axes[0].set_title('MAE & RMSE by Model'); axes[0].set_ylabel('µg/m³')
axes[0].tick_params(axis='x', rotation=45)
comparison.set_index('Model')['R2'].plot(kind='bar', color='seagreen', ax=axes[1])
axes[1].set_title('R² by Model'); axes[1].set_ylabel('R²')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 5. Feature Importances — Tree-Based Models

Decision Tree, Random Forest, and Gradient Boosting all expose `feature_importances_`. Plotting them side by side reveals which predictors the ensembles actually rely on — we expect the recent-history lag/rolling features to dominate.

In [ ]:
tree_models = [('Decision Tree', dt), ('Random Forest', rf), ('Gradient Boosting', gb)]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, model) in zip(axes, tree_models):
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values().tail(12)
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{name} — Top Feature Importances')
plt.tight_layout(); plt.show()

## 6. Time-Series Cross-Validation

Standard k-fold CV would shuffle rows and leak future data into earlier folds. We use `TimeSeriesSplit`, which always trains on past data and validates on the chronologically following block, on a representative subset of models.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = {}

cv_candidates = {
    'Linear Regression': ('scaled', lr),
    'Ridge':             ('scaled', ridge),
    'Decision Tree':     ('raw',    dt),
    'Random Forest':     ('raw',    rf),
    'Gradient Boosting': ('raw',    gb),
}

X_full_scaled = scaler.fit_transform(X)

for name, (kind, model) in cv_candidates.items():
    Xuse = X_full_scaled if kind == 'scaled' else X
    scores = cross_val_score(model, Xuse, y, cv=tscv, scoring='r2', n_jobs=-1)
    cv_scores[name] = scores
    print(f'{name:25s}  mean R² = {scores.mean():.4f}  (+/- {scores.std():.4f})')

In [ ]:
plt.figure(figsize=(11, 5))
plt.boxplot([cv_scores[k] for k in cv_scores], labels=list(cv_scores.keys()))
plt.ylabel('R²'); plt.title('Time-Series CV — R² by Model')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

## 7. Hyperparameter Tuning — Best Performer

We identify the top-performing model from the comparison table and run `GridSearchCV` with a `TimeSeriesSplit` CV (so tuning also respects the time order). The tuned estimator is then evaluated on the held-out test set.

In [ ]:
best_model_name = comparison.iloc[0]['Model']
print(f'Best model: {best_model_name}')

base_name = best_model_name.split(' (k=')[0] if 'KNN' in best_model_name else best_model_name
kind_best, best_obj = models[best_model_name]

if 'Random Forest' in base_name:
    param_grid = {'n_estimators': [200, 300], 'max_depth': [None, 20], 'min_samples_leaf': [1, 2]}
    tuner_model = RandomForestRegressor(random_state=42, n_jobs=-1)
    use_scaled = False
elif 'Gradient Boosting' in base_name:
    param_grid = {'n_estimators': [150, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [4, 5]}
    tuner_model = GradientBoostingRegressor(random_state=42)
    use_scaled = False
elif 'Decision Tree' in base_name:
    param_grid = {'max_depth': [8, 10, 15], 'min_samples_leaf': [1, 2, 5]}
    tuner_model = DecisionTreeRegressor(random_state=42)
    use_scaled = False
elif 'KNN' in base_name:
    param_grid = {'n_neighbors': [3, 5, 7, 9]}
    tuner_model = KNeighborsRegressor(n_jobs=-1)
    use_scaled = True
elif 'Ridge' in base_name:
    param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
    tuner_model = Ridge()
    use_scaled = True
else:
    param_grid = {'alpha': [0.01, 0.1, 1.0]}
    tuner_model = Lasso(max_iter=10000)
    use_scaled = True

Xtr_g = X_train_scaled if use_scaled else X_train
Xte_g = X_test_scaled  if use_scaled else X_test

grid = GridSearchCV(tuner_model, param_grid, cv=TimeSeriesSplit(n_splits=3),
                    scoring='r2', n_jobs=-1, verbose=0)
grid.fit(Xtr_g, y_train)
print('Best params:', grid.best_params_)
print('Best CV R²:', round(grid.best_score_, 4))

best_tuned = grid.best_estimator_
y_pred_tuned = best_tuned.predict(Xte_g)
tuned_label = f'{base_name} (Tuned)'
tuned_metrics = evaluate_model(tuned_label, y_test, y_pred_tuned)
results.append(tuned_metrics)

## 8. Residual Analysis — Tuned Model

A residuals-vs-predicted scatter plot and a histogram of residuals reveal whether errors are random and centred on zero (good) or show structure / heteroscedasticity. Pollution data typically shows larger errors during extreme smog episodes.

In [ ]:
plot_residuals(y_test.values, y_pred_tuned, tuned_label)
plt.show()

## 9. Prediction Example — Last Test Rows

We inspect the model's predictions on the most recent hours in the test set, where the time-series context is fully populated, and compare predicted vs actual PM2.5 with the absolute error.

In [ ]:
n_sample = 10
sample_X = Xte_g[-n_sample:] if isinstance(Xte_g, np.ndarray) else Xte_g.iloc[-n_sample:]
sample_preds = best_tuned.predict(sample_X)
sample_actual = y_test.values[-n_sample:]

pred_df = pd.DataFrame({
    'Datetime': df['Datetime'].iloc[split_idx:].values[-n_sample:],
    'Actual':    sample_actual.round(1),
    'Predicted': sample_preds.round(1),
    'AbsError':  np.abs(sample_actual - sample_preds).round(1),
})
pred_df

## 10. Final Summary

### Model Comparison (all models, sorted by R²)

In [ ]:
final = compare_models(results)
print(final.to_string(index=False))

### Key Takeaways

- **Recent-history features dominate**: `lag_1` (pollution one hour ago) and the 24-hour rolling mean are by far the strongest predictors — PM2.5 is highly persistent, so the recent past is the best guide to the present.
- **Tree ensembles win**: Random Forest and Gradient Boosting capture the non-linear interactions between weather, season, and recent pollution that linear models cannot, and lead the leaderboard.
- **Weather features add signal beyond persistence**: dew point, wind speed (`Iws`), and pressure contribute the meteorological context that explains *why* pollution rises or clears.
- **Chronological evaluation is honest**: by never shuffling and using `TimeSeriesSplit`, the reported metrics reflect genuine forecasting performance on unseen future hours.

### Next Steps

- Add longer lags (`lag_168` = same hour last week) and weather-lag interactions.
- Try gradient-boosting libraries (XGBoost / LightGBM) for further gains.
- Forecast multiple steps ahead instead of a single next-hour value.